### 4.1 Q-Learning（表格Q学习）

> **做什么**：维护Q表，通过epsilon-贪心策略探索，更新Q值  
> **经典案例**：FrozenLake冰湖问题（4x4网格，避开冰窟到达终点）


In [1]:
import numpy as np
import gymnasium as gym

# 创建FrozenLake环境（is_slippery=False使行动确定，便于学习）
env = gym.make('FrozenLake-v1', map_name="4x4", is_slippery=False)
n_states = env.observation_space.n   # 16个状态
n_actions = env.action_space.n       # 4个动作（上下左右）

# 初始化Q表：所有Q值设为0
Q = np.zeros((n_states, n_actions))

# 超参数
alpha = 0.8     # 学习率
gamma = 0.95    # 折扣因子
epsilon = 1.0   # 探索率（初始全探索）
n_episodes = 2000

for ep in range(n_episodes):
    state, _ = env.reset()
    done = False
    while not done:
        # epsilon-贪心策略：以epsilon概率随机探索，否则选最优动作
        if np.random.random() < epsilon:
            action = env.action_space.sample()  # 随机探索
        else:
            action = np.argmax(Q[state])        # 利用最优

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # Q-Learning更新：Q(s,a) <- Q(s,a) + alpha[r + gamma*max Q(s',a') - Q(s,a)]
        Q[state, action] += alpha * (
            reward + gamma * np.max(Q[next_state]) - Q[state, action]
        )
        state = next_state

    # 探索率衰减：逐渐减少探索，增加利用
    epsilon = max(0.01, epsilon * 0.995)

# 测试学到的策略
state, _ = env.reset()
done = False
total_reward = 0
while not done:
    action = np.argmax(Q[state])
    state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    total_reward += reward
print(f"测试总奖励: {total_reward} (1=成功到达终点)")
print(f"Q表最大值: {Q.max():.3f}")

测试总奖励: 1 (1=成功到达终点)
Q表最大值: 1.000


### 4.2 DQN（Deep Q-Network）

> **做什么**：用神经网络近似Q表，经验回放+目标网络稳定训练  
> **经典案例**：CartPole平衡小车


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque
import gymnasium as gym

device = torch.device('cpu')

# ====== 经验回放缓冲区 ======
class ReplayBuffer:
    def __init__(self, capacity=5000):
        self.buffer = deque(maxlen=capacity)
    def push(self, *transition):
        self.buffer.append(transition)
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return zip(*batch)
    def __len__(self):
        return len(self.buffer)

# ====== Q网络 ======
class QNet(nn.Module):
    def __init__(self, n_state, n_action):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_state, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, n_action)
        )
    def forward(self, x): return self.net(x)

env = gym.make('CartPole-v1')
n_state, n_action = env.observation_space.shape[0], env.action_space.n

policy_net = QNet(n_state, n_action).to(device)    # 策略网络（在线更新）
target_net = QNet(n_state, n_action).to(device)    # 目标网络（定期同步）
target_net.load_state_dict(policy_net.state_dict())
optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)
buffer = ReplayBuffer()

gamma = 0.99; epsilon = 1.0; batch_size = 64

for ep in range(200):
    state, _ = env.reset()
    total_reward = 0
    done = False
    while not done:
        # epsilon-贪心选动作
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                action = policy_net(torch.FloatTensor(state).to(device)).argmax().item()
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        buffer.push(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

        # 从经验回放采样训练
        if len(buffer) >= batch_size:
            s, a, r, s2, d = buffer.sample(batch_size)
            s = torch.FloatTensor(np.array(list(s))).to(device)
            a = torch.LongTensor(list(a)).unsqueeze(1).to(device)
            r = torch.FloatTensor(list(r)).to(device)
            s2 = torch.FloatTensor(np.array(list(s2))).to(device)
            d = torch.FloatTensor(list(d)).to(device)

            q_values = policy_net(s).gather(1, a).squeeze()        # Q(s,a)
            with torch.no_grad():
                q_next = target_net(s2).max(1)[0]                  # max Q(s',a')
                target = r + gamma * q_next * (1 - d)              # 目标值
            loss = nn.functional.mse_loss(q_values, target)
            optimizer.zero_grad(); loss.backward(); optimizer.step()

    # 定期同步目标网络
    if (ep+1) % 10 == 0:
        target_net.load_state_dict(policy_net.state_dict())
    epsilon = max(0.01, epsilon * 0.995)
    if (ep+1) % 20 == 0:
        print(f"Episode {ep+1}, Reward: {total_reward:.0f}, Epsilon: {epsilon:.3f}")

Episode 20, Reward: 11, Epsilon: 0.905
Episode 40, Reward: 11, Epsilon: 0.818
Episode 60, Reward: 16, Epsilon: 0.740
Episode 80, Reward: 28, Epsilon: 0.670
Episode 100, Reward: 59, Epsilon: 0.606
Episode 120, Reward: 41, Epsilon: 0.548
Episode 140, Reward: 126, Epsilon: 0.496
Episode 160, Reward: 107, Epsilon: 0.448
Episode 180, Reward: 13, Epsilon: 0.406
Episode 200, Reward: 47, Epsilon: 0.367


### 4.3 AutoGPT 概念与架构示意

> **做什么**：基于LLM的自主Agent，循环执行"思考->行动->观察"  
> **注意**：不写完整AutoGPT代码，仅给出架构示意+API调用模板


AutoGPT 架构示意
================

A|Thought(思考) LLM规划下一步| --> B|Action(行动) 调用工具/API| --> C|Observation(观察)获取执行结果| --> A

核心组件：
1. 记忆(Memory): 短期对话历史 + 长期向量检索
2. 规划(Planning): LLM拆解目标为子任务
3. 工具(Tools): 搜索、代码执行、文件读写等
4. 反馈(Feedback): 根据观察结果调整下一步

In [4]:
import requests
import json

response = requests.post("http://localhost:11434/api/generate", json={
    "model": "gemma:2b",
    "prompt": "用一句话解释什么是机器学习",
    "stream": False
})
print(json.loads(response.text)["response"])

机器学习是一种计算机科学技术，通过让计算机从数据中学习，从而做出新的预测或决策的过程。


In [2]:
import requests
import json
import re

# ========== 工具集 ==========
def search_web(query):
    """模拟搜索（你可以换成真实搜索API）"""
    return f"搜索结果：关于「{query}」，找到了相关文章..."

def calculator(expr):
    """计算器"""
    try:
        return str(eval(expr))
    except:
        return "计算错误"

def write_file(filename, content):
    """写文件"""
    with open(f"D:/JupyterNotebook/test/work2/{filename}", "w", encoding="utf-8") as f:
        f.write(content)
    return f"文件 {filename} 已保存"

TOOLS = {
    "search": search_web,
    "calculator": calculator,
    "write_file": write_file,
}

# ========== LLM 调用 ==========
def call_llm(prompt):
    resp = requests.post("http://localhost:11434/api/generate", json={
        "model": "gemma:2b",
        "prompt": prompt,
        "stream": False
    })
    return json.loads(resp.text)["response"]

# ========== Agent 循环 ==========
def agent_loop(goal, max_steps=5):
    history = [f"【目标】{goal}"]
    
    for step in range(max_steps):
        print(f"\n{'='*40}\n第 {step+1} 步")
        
        # 构造 prompt：让 LLM 选择下一步行动
        prompt = f"""你是一个AI助手，需要完成以下目标。

目标：{goal}

历史记录：
{chr(10).join(history[-10:])}

可用工具：search(query), calculator(expr), write_file(filename, content)
回复格式：
THOUGHT: <你的思考>
ACTION: <工具名>
ARGS: <参数>
或者 FINAL: <最终答案>

现在，决定下一步："""
        
        response = call_llm(prompt)
        print(f"LLM输出:\n{response}")
        
        history.append(f"Step {step+1}: {response}")
        
        # 解析动作
        if "FINAL:" in response:
            print(f"\n✅ 完成！最终答案：{response.split('FINAL:')[-1].strip()}")
            return
        
        action_match = re.search(r'ACTION:\s*(\w+)', response)
        args_match = re.search(r'ARGS:\s*(.+)', response)
        
        if action_match and args_match:
            tool_name = action_match.group(1).strip()
            args = args_match.group(1).strip()
            
            if tool_name in TOOLS:
                result = TOOLS[tool_name](args)
                print(f"工具结果: {result}")
                history.append(f"结果: {result}")
            else:
                print(f"未知工具: {tool_name}")
        else:
            print("解析失败，跳过")

# ========== 运行 ==========
agent_loop("计算 15*23+7 的结果，并把结果保存到 answer.txt", max_steps=5)


第 1 步
LLM输出:
**THOUGHT: 计算 15*23+7 的结果，并把结果保存到 answer.txt**

**ACTION: calculator(expr)**
**ARGS: 15 23 + 7**
**FINAL: 42**

✅ 完成！最终答案：42**
